In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import json, time, copy
from datetime import datetime

DATA_DIR = Path('../data')
RESULTS_DIR = Path('../data')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
all_df = pd.read_csv(DATA_DIR / 'features.csv')
print(f'Loaded {len(all_df)} rows, {all_df["page_id"].nunique()} pages')

# Derived features
TAG_MAP = {'body': 0, 'header': 1, 'nav': 2, 'main': 3, 'article': 4,
           'section': 5, 'footer': 6, 'aside': 7, 'form': 8}
all_df['parent_tag_idx'] = all_df['parent_tag'].map(TAG_MAP).fillna(0).astype(int)

tag_trans, dist_head = [], []
for _, group in all_df.groupby('page_id', sort=False):
    transitions = (group['parent_tag'] != group['parent_tag'].shift(1)).astype(int)
    transitions.iloc[0] = 0
    tag_trans.append(transitions)
    last_h = -1
    dists = []
    for i, (_, row) in enumerate(group.iterrows()):
        if row['is_heading'] == 1:
            last_h = i
        dists.append(i - last_h if last_h >= 0 else i)
    dist_head.append(pd.Series(dists, index=group.index))

all_df['tag_transition'] = pd.concat(tag_trans)
all_df['dist_to_heading'] = pd.concat(dist_head)

# 80/20 split
all_pages = all_df['page_id'].unique()
rng = np.random.RandomState(42)
rng.shuffle(all_pages)
split_idx = int(len(all_pages) * 0.8)
train_ids = set(all_pages[:split_idx])
test_ids = set(all_pages[split_idx:])

train_df = all_df[all_df['page_id'].isin(train_ids)].reset_index(drop=True)
test_df = all_df[all_df['page_id'].isin(test_ids)].reset_index(drop=True)

print(f'Train: {len(train_df)} rows, {len(train_ids)} pages')
print(f'Test:  {len(test_df)} rows, {len(test_ids)} pages')

In [ ]:
# All 56 features
ALL_FEATURES = [
    'position_pct', 'total_lines', 'depth',
    'ancestor_depth_ratio', 'parent_tag_idx',
    'in_header', 'in_nav', 'in_main', 'in_article',
    'in_footer', 'in_aside', 'in_form',
    'text_length', 'word_count', 'link_ratio',
    'is_link_only', 'is_heading', 'heading_level',
    'has_bold', 'is_list_item',
    'is_link_summary', 'is_cookie_summary',
    'punctuation_ratio', 'sentence_count', 'avg_word_length',
    'uppercase_ratio', 'number_ratio', 'type_token_ratio',
    'comma_count',
    'has_positive_class', 'has_negative_class', 'has_unlikely_class',
    'has_image', 'has_input',
    'is_button', 'has_aria_hidden', 'in_table', 'in_details',
    'mean_idf', 'max_idf', 'line_frequency',
    'word_novelty', 'word_novelty_sum',
    'cumulative_text_pct',
    'mean_text_length_w5', 'mean_link_ratio_w5',
    'block_size', 'block_text_density', 'block_link_density',
    'relative_position_in_block', 'sibling_text_variance',
    'style_group_size', 'style_group_link_density', 'style_group_mean_words',
    'tag_transition', 'dist_to_heading',
]

missing = [f for f in ALL_FEATURES if f not in all_df.columns]
assert not missing, f'Missing: {missing}'
assert len(ALL_FEATURES) == 56, f'Expected 56, got {len(ALL_FEATURES)}'
print(f'Feature pool: {len(ALL_FEATURES)}')

## Model and infrastructure

In [ ]:
HIDDEN = 128
NUM_LAYERS = 2
DROPOUT = 0.2
LR = 5e-4
BATCH_SIZE = 16
SEARCH_EPOCHS = 20
CONFIRM_EPOCHS = 30
SEEDS = [42, 123, 777]
N_STARTS = 20

n_content = train_df['label'].sum()
n_total = len(train_df)
POS_WEIGHT = (n_total - n_content) / n_content
print(f'Pos weight: {POS_WEIGHT:.3f}')


class PageDataset(Dataset):
    def __init__(self, pages):
        self.pages = pages
    def __len__(self):
        return len(self.pages)
    def __getitem__(self, idx):
        feats, labels, _, _, _ = self.pages[idx]
        return feats, labels


def collate_fn(batch):
    feats_list, labels_list = zip(*batch)
    max_len = max(f.shape[0] for f in feats_list)
    n_feats = feats_list[0].shape[1]
    batch_size = len(batch)
    feats_padded = torch.zeros(batch_size, max_len, n_feats)
    labels_padded = torch.zeros(batch_size, max_len)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)
    for i, (f, l) in enumerate(zip(feats_list, labels_list)):
        seq_len = f.shape[0]
        feats_padded[i, :seq_len] = f
        labels_padded[i, :seq_len] = l
        mask[i, :seq_len] = True
    return feats_padded, labels_padded, mask


class BiGRUModel(nn.Module):
    def __init__(self, n_features, hidden=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.proj = nn.Linear(n_features, hidden)
        self.gru = nn.GRU(
            hidden, hidden // 2, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        x = F.relu(self.proj(x))
        x, _ = self.gru(x)
        x = self.drop(x)
        return self.head(x).squeeze(2)


def compute_loss(logits, labels, mask):
    bce = F.binary_cross_entropy_with_logits(logits, labels, reduction='none')
    weights = torch.where(labels == 1, POS_WEIGHT, 1.0)
    bce_loss = (bce * weights * mask.float()).sum() / mask.sum()
    probs = torch.sigmoid(logits)
    probs_m = probs * mask.float()
    labels_m = labels * mask.float()
    inter = (probs_m * labels_m).sum(dim=1)
    union = (probs_m + labels_m - probs_m * labels_m).sum(dim=1)
    iou = (inter + 1) / (union + 1)
    iou_loss = 1 - iou.mean()
    return bce_loss + 1.0 * iou_loss


def df_to_pages(df, feature_cols, feat_mean, feat_std):
    pages = []
    for page_id, group in df.groupby('page_id', sort=False):
        group = group.sort_values('line_num')
        feats = group[feature_cols].values.astype(np.float32)
        feats = (feats - feat_mean) / feat_std
        labels = group['label'].values.astype(np.float32)
        pages.append((torch.from_numpy(feats), torch.from_numpy(labels),
                       group['line_num'].values, group['span_lines'].values, page_id))
    return pages


def evaluate_pages(all_preds, all_labels, pages, threshold=0.5):
    results = []
    for i, (preds, labels) in enumerate(zip(all_preds, all_labels)):
        pred_bin = (preds >= threshold).astype(int)
        label_bin = labels.astype(int)
        _, _, line_nums, span_lines, page_id = pages[i]
        pred_set, truth_set = set(), set()
        for j in range(len(pred_bin)):
            ln, span = int(line_nums[j]), int(span_lines[j])
            lines = set(range(ln, ln + span))
            if pred_bin[j]: pred_set |= lines
            if label_bin[j]: truth_set |= lines
        if not pred_set and not truth_set:
            iou, prec, rec = 1.0, 1.0, 1.0
        else:
            inter = len(pred_set & truth_set)
            union = len(pred_set | truth_set)
            iou = inter / union if union else 0.0
            prec = inter / len(pred_set) if pred_set else 0.0
            rec = inter / len(truth_set) if truth_set else 0.0
        results.append({'page_id': page_id, 'iou': iou, 'precision': prec, 'recall': rec})
    return results


print('Infrastructure ready.')

In [ ]:
def run_single(feature_cols, seed, num_epochs=SEARCH_EPOCHS):
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_feats = train_df[feature_cols].values.astype(np.float32)
    feat_mean = train_feats.mean(axis=0)
    feat_std = train_feats.std(axis=0)
    feat_std[feat_std == 0] = 1.0

    train_pages = df_to_pages(train_df, feature_cols, feat_mean, feat_std)
    test_pages = df_to_pages(test_df, feature_cols, feat_mean, feat_std)

    train_loader = DataLoader(
        PageDataset(train_pages), batch_size=BATCH_SIZE,
        shuffle=True, collate_fn=collate_fn,
        generator=torch.Generator().manual_seed(seed),
    )
    test_loader = DataLoader(
        PageDataset(test_pages), batch_size=BATCH_SIZE,
        shuffle=False, collate_fn=collate_fn,
    )

    model = BiGRUModel(len(feature_cols), hidden=HIDDEN,
                       num_layers=NUM_LAYERS, dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_iou = 0.0
    best_prec = 0.0
    best_rec = 0.0
    best_epoch = 0

    for epoch in range(1, num_epochs + 1):
        model.train()
        for feats, labels, mask in train_loader:
            feats, labels, mask = feats.to(DEVICE), labels.to(DEVICE), mask.to(DEVICE)
            logits = model(feats)
            loss = compute_loss(logits, labels, mask)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        scheduler.step()

        if epoch % 5 == 0 or epoch == num_epochs:
            model.eval()
            all_preds, all_labels_out = [], []
            with torch.no_grad():
                for feats, labels, mask in test_loader:
                    logits = model(feats.to(DEVICE))
                    probs = torch.sigmoid(logits).cpu()
                    for i in range(feats.shape[0]):
                        length = mask[i].sum().item()
                        all_preds.append(probs[i, :length].numpy())
                        all_labels_out.append(labels[i, :length].numpy())
            results = evaluate_pages(all_preds, all_labels_out, test_pages)
            mi = np.mean([r['iou'] for r in results])
            mp = np.mean([r['precision'] for r in results])
            mr = np.mean([r['recall'] for r in results])
            if mi > best_iou:
                best_iou, best_prec, best_rec, best_epoch = mi, mp, mr, epoch

    return {'iou': best_iou, 'prec': best_prec, 'rec': best_rec, 'epoch': best_epoch}


def run_experiment(feature_cols, num_epochs=SEARCH_EPOCHS):
    ious = []
    for seed in SEEDS:
        result = run_single(feature_cols, seed, num_epochs)
        ious.append(result['iou'])
    return {'iou': round(np.mean(ious), 4), 'std': round(np.std(ious), 4),
            'ious': [round(v, 4) for v in ious]}


print('run_experiment() ready (3-seed).')

## Baseline: all 56 features

In [ ]:
t0 = time.time()
baseline_result = run_experiment(ALL_FEATURES)
elapsed = time.time() - t0
print(f'Baseline (56 feat): IoU={baseline_result["iou"]:.4f} +/- {baseline_result["std"]:.4f}')
print(f'  Seeds: {baseline_result["ious"]}')
print(f'  Time per 3-seed eval: {elapsed:.0f}s')
print(f'  Est. backward round (56 features): ~{elapsed * 56 / 60:.0f} min')
print(f'  Est. full search (20 starts): ~{elapsed * 56 * 4 * 20 / 3600:.0f} hours')

## Multi-start backward/forward search

20 random starting sets with diverse sizes (10–50 features).
Each start runs backward/forward cycling until convergence.
Progress saved after each start converges — resume-safe.

In [ ]:
RESULTS_PATH = RESULTS_DIR / 'bigru_feature_search_56_results.json'


def save_progress(converged, global_log):
    """Save after each start converges for crash recovery."""
    with open(RESULTS_PATH, 'w') as f:
        json.dump({
            'timestamp': datetime.now().isoformat(),
            'config': {'hidden': HIDDEN, 'num_layers': NUM_LAYERS, 'dropout': DROPOUT,
                       'lr': LR, 'search_epochs': SEARCH_EPOCHS, 'seeds': SEEDS,
                       'n_starts': N_STARTS},
            'baseline': {'n_features': 56, **baseline_result},
            'converged': converged,
            'global_log': global_log,
        }, f, indent=2)


def load_progress():
    """Load previously saved progress for resume."""
    if RESULTS_PATH.exists():
        with open(RESULTS_PATH) as f:
            data = json.load(f)
        return data.get('converged', []), data.get('global_log', [])
    return [], []


def backward_forward_cycle(start_features, start_id, verbose=True):
    """Run backward/forward cycling from a starting set until convergence."""
    current_set = list(start_features)
    current_iou = run_experiment(current_set)['iou']
    log = []

    if verbose:
        print(f'  Start {start_id}: {len(current_set)} features, IoU={current_iou:.4f}')

    cycle = 0
    while True:
        cycle += 1
        prev_set = set(current_set)

        # --- Backward ---
        while len(current_set) > 1:
            best_drop, best_drop_iou = None, 0
            for feat in current_set:
                candidate = [f for f in current_set if f != feat]
                result = run_experiment(candidate)
                if result['iou'] > best_drop_iou:
                    best_drop_iou = result['iou']
                    best_drop = feat
            if best_drop_iou > current_iou:
                current_set = [f for f in current_set if f != best_drop]
                current_iou = best_drop_iou
                log.append(f'C{cycle} drop {best_drop} -> {len(current_set)}f IoU={current_iou:.4f}')
                if verbose:
                    print(f'    C{cycle} drop {best_drop:30s} -> {len(current_set)}f IoU={current_iou:.4f}')
            else:
                break

        # --- Forward ---
        remaining = [f for f in ALL_FEATURES if f not in current_set]
        while remaining:
            best_add, best_add_iou = None, 0
            for feat in remaining:
                candidate = current_set + [feat]
                result = run_experiment(candidate)
                if result['iou'] > best_add_iou:
                    best_add_iou = result['iou']
                    best_add = feat
            if best_add_iou > current_iou:
                current_set.append(best_add)
                remaining.remove(best_add)
                current_iou = best_add_iou
                log.append(f'C{cycle} add  {best_add} -> {len(current_set)}f IoU={current_iou:.4f}')
                if verbose:
                    print(f'    C{cycle} add  {best_add:30s} -> {len(current_set)}f IoU={current_iou:.4f}')
            else:
                break

        if set(current_set) == prev_set:
            break

    return {
        'start_id': start_id,
        'n_start': len(start_features),
        'features': sorted(current_set),
        'n_features': len(current_set),
        'iou': current_iou,
        'cycles': cycle,
        'log': log,
    }


print('Search functions ready.')

In [ ]:
# Generate 20 diverse starting sets
# - Start 0: all 56 (full backward)
# - Starts 1-5: large (35-50 features)
# - Starts 6-12: medium (20-34 features)
# - Starts 13-19: small (10-19 features)

start_rng = np.random.RandomState(2026)
starting_sets = []

# Start 0: all features
starting_sets.append(list(ALL_FEATURES))

# Starts 1-5: large random subsets
for _ in range(5):
    n = start_rng.randint(35, 51)
    idx = start_rng.choice(56, size=n, replace=False)
    starting_sets.append([ALL_FEATURES[i] for i in sorted(idx)])

# Starts 6-12: medium random subsets
for _ in range(7):
    n = start_rng.randint(20, 35)
    idx = start_rng.choice(56, size=n, replace=False)
    starting_sets.append([ALL_FEATURES[i] for i in sorted(idx)])

# Starts 13-19: small random subsets
for _ in range(7):
    n = start_rng.randint(10, 20)
    idx = start_rng.choice(56, size=n, replace=False)
    starting_sets.append([ALL_FEATURES[i] for i in sorted(idx)])

print(f'Generated {len(starting_sets)} starting sets:')
for i, s in enumerate(starting_sets):
    print(f'  Start {i:2d}: {len(s):2d} features')

In [ ]:
# Run multi-start search (resume-safe)
converged, global_log = load_progress()
done_ids = {c['start_id'] for c in converged}

if done_ids:
    print(f'Resuming: {len(done_ids)}/{N_STARTS} starts already done')
else:
    print(f'Starting fresh: {N_STARTS} starts')

t_total = time.time()

for start_id in range(N_STARTS):
    if start_id in done_ids:
        continue

    print(f'\n{"=" * 70}')
    print(f'START {start_id}/{N_STARTS-1} ({len(starting_sets[start_id])} initial features)')
    print(f'Completed so far: {len(converged)}/{N_STARTS}')
    print(f'Best so far: {max((c["iou"] for c in converged), default=0):.4f}')
    print(f'{"=" * 70}')

    t_start = time.time()
    result = backward_forward_cycle(starting_sets[start_id], start_id)
    result['time_s'] = round(time.time() - t_start)

    converged.append(result)
    global_log.append({
        'start_id': start_id,
        'n_start': result['n_start'],
        'n_final': result['n_features'],
        'iou': result['iou'],
        'time_s': result['time_s'],
    })

    print(f'  Converged: {result["n_features"]} features, IoU={result["iou"]:.4f} ({result["time_s"]}s, {result["cycles"]} cycles)')

    save_progress(converged, global_log)
    print(f'  Progress saved.')

total_time = time.time() - t_total
print(f'\n{"=" * 70}')
print(f'All {N_STARTS} starts complete in {total_time/3600:.1f} hours')

## Results overview

In [ ]:
# Sort converged sets by IoU
ranked = sorted(converged, key=lambda x: x['iou'], reverse=True)

print(f'{"Start":>5} {"Init":>5} {"Final":>5} {"IoU":>8} {"Cycles":>6} {"Time":>8}')
print('-' * 45)
for r in ranked:
    print(f'{r["start_id"]:5d} {r["n_start"]:5d} {r["n_features"]:5d} {r["iou"]:8.4f} {r["cycles"]:6d} {r["time_s"]:7d}s')

ious = [r['iou'] for r in ranked]
print(f'\nMedian IoU: {np.median(ious):.4f}')
print(f'Mean IoU:   {np.mean(ious):.4f}')
print(f'Range:      {min(ious):.4f} — {max(ious):.4f}')
print(f'Feature set sizes: {sorted(set(r["n_features"] for r in ranked))}')

# Feature frequency across converged sets
from collections import Counter
feat_freq = Counter()
for r in converged:
    feat_freq.update(r['features'])

print(f'\nFeature frequency across {len(converged)} converged sets:')
print(f'{"Feature":30s} {"Count":>5} {"Pct":>6}')
print('-' * 45)
for feat, count in feat_freq.most_common():
    pct = count / len(converged) * 100
    print(f'{feat:30s} {count:5d} {pct:5.0f}%')

## Confirm top 3 sets (30 epochs)

In [ ]:
# Deduplicate by feature set (same features may appear from different starts)
seen = set()
unique_ranked = []
for r in ranked:
    key = tuple(sorted(r['features']))
    if key not in seen:
        seen.add(key)
        unique_ranked.append(r)

top_n = min(3, len(unique_ranked))
print(f'Confirming top {top_n} unique sets with {CONFIRM_EPOCHS} epochs, 3 seeds each')
print('=' * 70)

confirmed = []
for rank, r in enumerate(unique_ranked[:top_n], 1):
    feats = r['features']
    print(f'\n--- Rank {rank}: {len(feats)} features (search IoU={r["iou"]:.4f}, start {r["start_id"]}) ---')

    seed_results = []
    for seed in SEEDS:
        t_seed = time.time()
        result = run_single(feats, seed, num_epochs=CONFIRM_EPOCHS)
        elapsed = time.time() - t_seed
        seed_results.append(result)
        print(f'  seed={seed}  IoU={result["iou"]:.4f}  P={result["prec"]:.3f}  R={result["rec"]:.3f}  ep={result["epoch"]}  ({elapsed:.0f}s)')

    confirm_ious = [sr['iou'] for sr in seed_results]
    confirm_mean = np.mean(confirm_ious)
    confirm_std = np.std(confirm_ious)
    print(f'  Confirmed: IoU={confirm_mean:.4f} +/- {confirm_std:.4f}')

    confirmed.append({
        'rank': rank,
        'start_id': r['start_id'],
        'features': feats,
        'n_features': len(feats),
        'search_iou': r['iou'],
        'confirmed_iou': round(confirm_mean, 4),
        'confirmed_std': round(confirm_std, 4),
        'seed_results': seed_results,
    })

## Final summary

In [ ]:
print('=' * 70)
print('FEATURE SEARCH RESULTS — 56 FEATURES, 20 STARTS')
print('=' * 70)

print(f'\nBaseline (56 feat): IoU={baseline_result["iou"]:.4f} +/- {baseline_result["std"]:.4f}')

# Best confirmed set
best = max(confirmed, key=lambda x: x['confirmed_iou'])
print(f'\nBest confirmed ({best["n_features"]} feat): IoU={best["confirmed_iou"]:.4f} +/- {best["confirmed_std"]:.4f}')
print(f'  Delta vs baseline: {best["confirmed_iou"] - baseline_result["iou"]:+.4f}')
print(f'  From start {best["start_id"]} (search IoU={best["search_iou"]:.4f})')

dropped = sorted(set(ALL_FEATURES) - set(best['features']))
print(f'\nDropped ({len(dropped)}):')
for f in dropped:
    print(f'  - {f}')
if not dropped:
    print('  (none)')

print(f'\nFinal feature set ({best["n_features"]}):')
for i, f in enumerate(sorted(best['features']), 1):
    print(f'  {i:2d}. {f}')

# All confirmed results
print(f'\nAll confirmed sets:')
for c in sorted(confirmed, key=lambda x: x['confirmed_iou'], reverse=True):
    print(f'  Rank {c["rank"]}: {c["n_features"]} feat, IoU={c["confirmed_iou"]:.4f} +/- {c["confirmed_std"]:.4f} (start {c["start_id"]})')

# Save final results
final_path = RESULTS_DIR / 'bigru_feature_search_56_final.json'
with open(final_path, 'w') as f:
    json.dump({
        'timestamp': datetime.now().isoformat(),
        'baseline': {'n_features': 56, **baseline_result},
        'best': {
            'features': sorted(best['features']),
            'n_features': best['n_features'],
            'confirmed_iou': best['confirmed_iou'],
            'confirmed_std': best['confirmed_std'],
        },
        'confirmed': confirmed,
        'converged': converged,
        'feature_frequency': dict(feat_freq.most_common()),
    }, f, indent=2)
print(f'\nSaved to {final_path}')